# 🧠 LangGraph Single Agent Implementation Training

Welcome to the hands-on course on building single AI agents using **LangGraph**. This notebook will guide you through concepts, code, and exercises.

---

### 📚 What You'll Learn in This Notebook

- **TypedDict vs dict** — Why type-safe state matters in agent workflows
- **Sequential Graphs** — Building linear, multi-step processing pipelines
- **Conditional Graphs** — Routing execution based on runtime state values
- **Looping Graphs** — Creating iterative workflows with termination conditions
- **Memory-Enabled Agents** — Adding short-term and long-term memory to your agents

### 🔑 Key Concepts

- **StateGraph**: The core LangGraph abstraction — a directed graph where nodes read/write to a shared state object
- **Nodes**: Python functions that transform state — each node receives the current state and returns an updated state
- **Edges**: Connections between nodes that define execution flow (sequential, conditional, or looping)
- **Compilation**: Turning your graph definition into a runnable application

### ⚡ Prerequisites

- Basic Python knowledge (functions, dictionaries, type hints)
- LangGraph installed (`pip install langgraph`)

---

Let's get started! 🚀

In [ ]:
# 📦 Install LangGraph if not already installed
# LangGraph is a framework for building stateful, multi-step agent workflows as directed graphs
!pip install langgraph

## 🔧 Using TypedDict Instead of dict

### Why This Matters

In LangGraph, the **state** is the central data structure that flows between nodes. Using Python's built-in `dict` works, but it has a major limitation: **no type checking**. You can accidentally put any value under any key, leading to hard-to-debug runtime errors.

**TypedDict** solves this by:
- Declaring **expected keys** and their **types** upfront
- Enabling IDE autocompletion and type-checker support
- Making your agent state self-documenting — anyone reading your code knows exactly what data flows through the graph

### 🔑 Key Takeaway
> Always use `TypedDict` to define your agent state in LangGraph. It acts as a **contract** that every node in your graph must respect.

In [ ]:
# 📝 Example: A regular Python dict — no type safety
# You can assign any type to any key without warnings or errors
my_llms = {"openai": "gpt-4", "anthropic": "claude-2"}
my_llms  # Works fine, but nothing prevents you from adding {"openai": 12345} later

In [ ]:
# ⚠️ Problem with plain dicts: They don't enforce data types on their elements
# For example, you could accidentally write my_llms["openai"] = 999 — no error!
# Solution: Use TypedDict to define a schema with enforced key names and value types
# This is especially important in LangGraph where state consistency between nodes is critical

In [ ]:
# ✅ TypedDict: The type-safe alternative to plain dicts
from typing import TypedDict

# Define a TypedDict class — this acts like a schema for your dictionary
# Each field has a declared type that type checkers can validate
class LLMs(TypedDict):
    openai: str              # Must be a string (e.g., "gpt-4")
    anthropic: list[str]     # Must be a list of strings (e.g., ["claude-2", "claude-opus-3"])

# Create an instance — keys and value types are now documented and enforceable
my_llms_typed = LLMs(openai="gpt-4", anthropic=["claude-2", "claude-opus-3"])
my_llms_typed  # Same dict-like behavior, but with type safety!

# 💡 In LangGraph, TypedDict defines your AgentState — the shared data contract between all nodes

# 🔀 Multiple Inputs Graph & Sequential Graph

### 📚 What You'll Learn

- How to define an **AgentState** with multiple fields using `TypedDict`
- How to create **node functions** that read from and write to the shared state
- How to wire nodes together in a **sequential (linear) pipeline**
- How to **compile** the graph and **invoke** it with initial state

### 🧠 Key Concepts

- **AgentState**: A `TypedDict` that acts as the shared memory between all nodes in your graph
- **Node Function**: A Python function that takes the current state as input, modifies it, and returns the updated state
- **Sequential Flow**: Nodes execute one after another — `Node A → Node B → Node C` — each building on the previous node's output
- **Entry Point / Finish Point**: Where the graph starts and ends execution

### 🎯 Why This Matters

Sequential graphs are the simplest LangGraph pattern. They're perfect for **multi-step processing pipelines** where each step transforms the data in order — like an assembly line where each station adds something to the product.

---

In [ ]:
# 📦 Import the building blocks for our first graph
from typing import TypedDict       # For defining type-safe state schemas
from langgraph.graph import StateGraph  # The core graph class that manages node execution

print("✅ Imports loaded successfully!")

In [ ]:
# 📋 Step 1: Define the AgentState — the shared data contract for our graph
# Every node in the graph will receive this state and can read/modify its fields
# Think of AgentState as a "clipboard" that gets passed from node to node

class AgentState(TypedDict):
    name: str          # The user's name — input field
    yoe: str           # Years of experience — input field
    final: str         # The output message — built up by nodes as they execute
    skills: list[str]  # A list of skills — input field (used in the exercise)

# 💡 Notice: We declare ALL fields upfront, even if some nodes don't use all of them
# This ensures every node knows the complete shape of the state it will receive
print("✅ AgentState defined with 4 fields: name, yoe, final, skills")

In [ ]:
# 🔧 Step 2: Define the Node Functions
# Each node is a regular Python function that:
#   - Takes the current AgentState as input
#   - Modifies one or more fields
#   - Returns the updated AgentState
# IMPORTANT: Input type = AgentState, Output type = AgentState (always!)

def first_node(state: AgentState) -> AgentState:
    """
    This is the first node in the graph.
    It reads the 'name' field and creates a personalized greeting in 'final'.
    """
    # Read the 'name' field from state and build a greeting string
    state["final"] = f'Hi {state["name"]}!'
    # Return the modified state — the next node will receive this updated version
    return state

def second_node(state: AgentState) -> AgentState:
    """
    This is the second node in the graph.
    It APPENDS to the 'final' field (built by the first node) with experience info.
    """
    # Notice: We're concatenating to state["final"], not overwriting it
    # This demonstrates how sequential nodes can build upon each other's work
    state["final"] = state["final"] + f' You have {state["yoe"]} years of experience!'
    return state

print("✅ Node functions defined: first_node (greeting) → second_node (experience)")

### 🏗️ Step 3: Building the Graph

Now we wire everything together. Building a LangGraph involves 5 steps:

1. **Create** a `StateGraph` with your state schema
2. **Add nodes** — register your functions with meaningful names
3. **Set entry point** — which node runs first
4. **Add edges** — connect nodes in sequence
5. **Set finish point** and **compile** — finalize the graph into a runnable app

> ⚠️ **Note**: Successful compilation doesn't mean the graph will run successfully — it only validates the graph structure (nodes and edges), not the logic inside your node functions.

In [ ]:
# 🏗️ Build the Sequential Graph — wiring nodes together in a linear pipeline

# Step 1: Create a StateGraph, passing our AgentState schema
# The StateGraph uses this schema to validate state flowing between nodes
graph = StateGraph(AgentState)

# Step 2: Register node functions with descriptive names
# These names are used when defining edges and will appear in visualizations
graph.add_node("Greeting", first_node)      # Register first_node as "Greeting"
graph.add_node("Experience", second_node)    # Register second_node as "Experience"

# Step 3: Define the execution flow
graph.set_entry_point("Greeting")            # "Greeting" is the first node to execute
graph.add_edge("Greeting", "Experience")     # After "Greeting", flow to "Experience"
graph.set_finish_point("Experience")         # "Experience" is the last node — graph ends here

# Step 4: Compile the graph into a runnable application
# Compilation validates the graph structure (all edges connect to valid nodes, etc.)
app = graph.compile()

print("✅ Graph compiled successfully! Flow: START → Greeting → Experience → END")

### 🎨 Visualizing the Graph

LangGraph can render your graph as a visual diagram using Mermaid. This is incredibly useful for:
- **Debugging** — verifying your edges connect the right nodes
- **Documentation** — showing others how your agent works
- **Validation** — confirming the flow matches your mental model

In [ ]:
# 🎨 Visualize the compiled graph — see the flow as a diagram
# get_graph() returns the graph structure, draw_mermaid_png() renders it as an image
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))
# 👆 You should see: __start__ → Greeting → Experience → __end__

### 🚀 Step 5: Invoking the Graph

Now let's run our compiled graph! We pass an **initial state dictionary** with values for the input fields (`name` and `yoe`). The graph will execute each node in sequence and return the final state with all fields populated.

In [ ]:
# 🚀 Step 5: Invoke the graph with initial state
# app.invoke() runs the entire graph from START to END
# We pass a dictionary matching our AgentState schema — only input fields need values
result = app.invoke({"name": "Mahendra", "yoe": 9})

# The result contains the FINAL state after all nodes have executed
print("📤 Full result state:", result)

In [ ]:
# 🎯 Extract just the 'final' field — the combined output built by both nodes
# first_node set: "Hi Mahendra!"
# second_node appended: " You have 9 years of experience!"
print("📝 Final message:", result["final"])

### ✅ Sequential Graph — Section Summary

You just built your first LangGraph application! Here's what happened:

1. **Defined** `AgentState` with `TypedDict` — established the data contract
2. **Created** two node functions — each transforms a piece of the state
3. **Wired** them sequentially — `Greeting → Experience`
4. **Compiled** and **invoked** — the graph executed each node in order

> 🧠 **Remember**: In a sequential graph, every node runs exactly once, in the order defined by edges. The state flows like water through a pipe — each node adds something to it.

---

## 🧪 Exercise: Sequential Graph

### Your Task

Build a **3-node sequential graph** that processes user profile information:

1. Accept a user's `name`, `yoe`, and a list of their `skills`
2. Pass the state through **three nodes** that:
    - **First node**: Personalizes the `name` field with a greeting
    - **Second node**: Describes the user's `yoe` (years of experience)
    - **Third node**: Lists the user's skills in a formatted string
3. The `final` output in the result field should be a combined message

### 💡 Hints

- Reuse the same `AgentState` schema defined above
- Each node should **append** to the `final` field (don't overwrite!)
- Remember the pattern: `graph.add_edge("NodeA", "NodeB")` to connect nodes

![Exercise_Sequential_graph](Exercise_Sequential_graph.png)

# 🔀 Conditional Graph

### 📚 What You'll Learn

- How to create **branching logic** in your graph based on runtime state
- How to use `add_conditional_edges()` to route execution dynamically
- How to implement a **router pattern** using a passthrough node
- The difference between `START`/`END` constants and `set_entry_point()`/`set_finish_point()`

### 🧠 Key Concepts

- **Conditional Edge**: An edge that evaluates a function at runtime to decide which node to execute next
- **Router Node**: A passthrough node whose only job is to be the source of conditional edges
- **Path Function**: A function that examines the state and returns a string indicating which path to take
- **Path Map**: A dictionary that maps the path function's return values to actual node names

### 🎯 Why This Matters

Real-world agents rarely follow a straight line. They need to **make decisions** — should I search the web or query the database? Should I escalate to a human or respond automatically? Conditional graphs let your agent choose different execution paths based on the current state.

### 🔄 Pattern: Router → Conditional Edges

```
START → Router → (if "add") → add_node → END
                 (if "subtract") → subtract_node → END
```

---

In [ ]:
# 📦 Import building blocks — note the new START and END constants!
# START and END are special sentinel nodes provided by LangGraph
# They replace set_entry_point() and set_finish_point() for a more explicit style
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("✅ Imports loaded — using START/END constants for explicit graph boundaries")

In [ ]:
# 📋 Define the AgentState for our calculator graph
# This state carries two numbers, an operation choice, and the result

class AgentState(TypedDict):
    number1: int          # First operand (e.g., 10)
    operation: str        # Operation to perform: "add" or "subtract" — this drives the conditional routing!
    number2: int          # Second operand (e.g., 4)
    finalNumber: int      # The computed result — set by whichever operation node runs

# 💡 The 'operation' field is the KEY decision variable — the router will inspect this
#    to decide whether to route to the addition or subtraction node
print("✅ AgentState defined for conditional calculator graph")

In [ ]:
# ➕ Operation Node 1: Addition
# This node runs ONLY when the router sends execution down the "add" path

def adder(state: AgentState) -> AgentState:
    """
    This node adds 2 numbers.
    Reads number1 and number2 from state, stores the sum in finalNumber.
    """
    print(f"  📥 adder received state: {state}")
    # Perform the addition and store the result
    state["finalNumber"] = state["number1"] + state["number2"]
    return state

print("✅ adder node function defined")

In [ ]:
# ➖ Operation Node 2: Subtraction
# This node runs ONLY when the router sends execution down the "subtract" path

def subtractor(state: AgentState) -> AgentState:
    """
    This node subtracts 2 numbers.
    Reads number1 and number2 from state, stores the difference in finalNumber.
    """
    print(f"  📥 subtractor received state: {state}")
    # Perform the subtraction and store the result
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

print("✅ subtractor node function defined")

In [ ]:
# 🔀 The Path Function — this is the BRAIN of conditional routing
# It examines the current state and returns a STRING that maps to a node name
# IMPORTANT: This function does NOT modify state — it only READS state to make a decision

def decide_next_node(state: AgentState) -> str:
    """
    This function decides which operation to perform based on state['operation'].
    Returns a string key that gets mapped to a node name via the path_map.
    """
    if state["operation"] == "add":
        return "addition_operation"       # This string maps to "add_node" in the path_map
    elif state["operation"] == "subtract":
        return "subtraction_operation"    # This string maps to "subtract_node" in the path_map
    else:
        raise ValueError("Invalid operation")  # Fail loudly if we get an unexpected value
    
# ⚠️ Notice: No state is returned here! This is NOT a node function.
# It's a ROUTING function — it returns a string, not a state dict.
print("✅ Router function defined — routes 'add' → addition_operation, 'subtract' → subtraction_operation")

### 🏗️ Building the Conditional Graph

Now we assemble the pieces. Pay close attention to `add_conditional_edges()` — this is the new concept:

- **`source`**: The node AFTER which the routing decision happens
- **`path`**: The function that examines state and returns a routing key string
- **`path_map`**: Maps routing key strings to actual node names

> 💡 **Router Pattern**: We use a passthrough node (`lambda state: state`) as the source for conditional edges. This keeps the routing logic cleanly separated from the data processing logic.

In [ ]:
# 🏗️ Build the Conditional Graph — this is where the magic happens!

graph = StateGraph(AgentState)

# Register the operation nodes — these are the "branches" of our conditional
graph.add_node("add_node", adder)          # The addition branch
graph.add_node("subtract_node", subtractor)  # The subtraction branch

# Register a ROUTER node — a passthrough that doesn't modify state
# Its only purpose is to serve as the "source" for conditional edges
# lambda state: state means "pass the state through unchanged"
graph.add_node("router", lambda state: state)

# Wire the graph:
# 1. START → router (always go to the router first)
graph.add_edge(START, "router")

# 2. router → (conditional) → add_node OR subtract_node
#    add_conditional_edges takes:
#      - source: the node AFTER which the decision is made
#      - path: the function that returns a routing key
#      - path_map: maps routing keys to actual node names
graph.add_conditional_edges(
    source="router",              # After this node, make a decision
    path=decide_next_node,        # Call this function to get the routing key
    path_map={
        # Format → routing_key: node_name
        "addition_operation": "add_node",        # If path returns "addition_operation" → go to "add_node"
        "subtraction_operation": "subtract_node"  # If path returns "subtraction_operation" → go to "subtract_node"
    }
)

# 3. Both branches lead to END (no further processing needed)
graph.add_edge("add_node", END)
graph.add_edge("subtract_node", END)

# Compile the graph
app = graph.compile()

print("✅ Conditional graph compiled! Flow: START → router → (add_node | subtract_node) → END")

In [ ]:
# 🎨 Visualize the conditional graph — you should see the branching structure!
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))
# 👆 You should see: __start__ → router → (add_node | subtract_node) → __end__
# The diamond shape on "router" indicates a conditional edge

### 🚀 Running the Conditional Graph

Let's test our calculator! We'll set `operation="subtract"` — the router should detect this and route to the `subtract_node`, skipping `add_node` entirely.

In [ ]:
# 🚀 Create the initial state using the AgentState constructor
# We set operation="subtract" — so the router should send us to the subtractor node
initial_state = AgentState(
    number1=10,            # First operand
    operation="subtract",  # This tells the router which path to take
    number2=4,             # Second operand
    finalNumber=0,         # Will be computed by the operation node
)

print(f"📥 Initial state prepared: {initial_state['number1']} {initial_state['operation']} {initial_state['number2']}")

In [ ]:
# 🏃 Invoke the graph with our initial state
# The router will check operation="subtract" and route to subtract_node
result = app.invoke(initial_state)
print(f"📤 Result: {result}")
print(f"🎯 10 - 4 = {result['finalNumber']}")

In [ ]:
# 💡 Alternative: You can also pass a plain dict instead of an AgentState instance
# LangGraph accepts any dict that matches the TypedDict schema
# This is more concise for quick testing
result = app.invoke({"number1": 10, "operation": "subtract", "number2": 4, "finalNumber": 0})
print(f"📤 Result (using plain dict): {result}")
print(f"🎯 10 - 4 = {result['finalNumber']}")

### ✅ Conditional Graph — Section Summary

You just built a graph that makes **runtime decisions**! Here's the pattern:

1. **Router node** — a passthrough node that serves as the decision point
2. **Path function** — examines state and returns a routing key (NOT a node function — returns a string, not state!)
3. **Path map** — translates routing keys to actual node names
4. **Conditional edges** — `add_conditional_edges(source, path, path_map)` wires it all together

> 🧠 **Key difference from sequential**: In a conditional graph, **not every node runs**. Only the node selected by the path function executes. The other branch is skipped entirely.

---

## 🧪 Exercise: Conditional Graph

### Your Task

Build an **extended calculator graph** with **two conditional routing stages**!

**Your task:**

- Make the graph shown below! You will need to make use of **2 conditional edges**!

**Input**: 
```python
initial_state = AgentState(
    number1=10, operation="-", number2=5, 
    number3=7, number4=2, operation2="+", 
    finalNumber=0, finalNumber2=0
)
```

### 💡 Hints
- You'll need an expanded `AgentState` with additional fields (`number3`, `number4`, `operation2`, `finalNumber2`)
- Create **two router nodes** — one for each conditional routing decision
- Each router examines a different `operation` field in the state
- The pattern is the same as above, just applied twice in sequence

![Exercise_Conditional_graph](Exercise_Conditional_graph.png)

# 🔄 Looping Graph

### 📚 What You'll Learn

- How to create **self-loops** — nodes that can re-execute themselves
- How to implement a **termination condition** to prevent infinite loops
- How to use a **counter** in state to track iterations
- The pattern: `conditional_edge` with one path pointing back to the same node

### 🧠 Key Concepts

- **Self-Loop**: A conditional edge where one path leads back to the current node, creating a cycle
- **Termination Condition**: A check in the path function that eventually returns `"exit"` instead of `"loop"`
- **Counter Pattern**: Using a state field (e.g., `counter`) to track how many times the loop has executed
- **Loop vs Exit**: The path function returns `"loop"` to continue or `"exit"` to break out

### 🎯 Why This Matters

Many agent tasks require **iteration** — retry logic, accumulating data until a threshold is met, polling for results, or running a conversation loop. Looping graphs let you express these patterns cleanly.

### 🔄 Pattern: Loop Until Done

```
START → greeting → random_number ←──┐
                        │            │
                   (counter < 5?) ───┘ "loop"
                        │
                   (counter >= 5) → END  "exit"
```

---

In [ ]:
# 📦 Import everything we need for the looping graph
from langgraph.graph import StateGraph, START, END  # Graph building blocks
from typing import TypedDict, Dict, List            # Type annotations
import random                                        # For generating random numbers

print("✅ Imports loaded for looping graph")

In [ ]:
# 📋 Define the AgentState for our looping graph
# The key field here is 'counter' — it controls when the loop terminates

class AgentState(TypedDict):
    name: str           # User's name — set once by the greeting node
    number: List[int]   # Accumulator list — each loop iteration appends a random number
    counter: int        # Loop counter — incremented each iteration, checked by termination condition

# 💡 The 'number' list grows with each iteration — this is how the loop "accumulates" results
# 💡 The 'counter' field is the termination guard — without it, the loop would run forever!
print("✅ AgentState defined with loop counter and accumulator list")

In [ ]:
# 🔧 Define three functions: a setup node, a loop body node, and a termination check

def greeting_node(state: AgentState) -> AgentState:
    """
    SETUP NODE: Runs once at the beginning.
    Greets the user and initializes the counter.
    """
    # Personalize the greeting — modifies the name field
    state["name"] = f"Hello {state['name']}!"
    # Initialize counter to 0 (handles edge case of negative initial values)
    state["counter"] = 0
    print(f"  👋 Greeting: {state['name']}, counter initialized to {state['counter']}")
    return state

def random_number_node(state: AgentState) -> AgentState:
    """
    LOOP BODY NODE: Runs on each iteration.
    Generates a random number and appends it to the accumulator list.
    Increments the counter to track how many iterations have occurred.
    """
    print(f"  🔄 Loop iteration {state['counter'] + 1} — state at start: {state}")
    # Append a random number (1-10) to the accumulator list
    state["number"].append(random.randint(1, 10))
    # Increment the counter — this is critical for the termination condition!
    state["counter"] += 1
    print(f"  ✅ Generated number, counter now = {state['counter']}, numbers = {state['number']}")
    return state

def should_continue_node(state: AgentState) -> str:
    """
    TERMINATION CHECK: Called after each iteration of random_number_node.
    Returns "loop" to continue or "exit" to stop.
    This is a PATH FUNCTION — returns a string, not a state!
    """
    if state["counter"] < 5:
        # Counter hasn't reached threshold — keep looping!
        print(f"  🔄 Counter = {state['counter']} (< 5) → continuing loop")
        return "loop"   # This maps to "random_number" node (self-loop)
    else:
        # Counter reached threshold — exit the loop
        print(f"  🏁 Counter = {state['counter']} (>= 5) → exiting loop")
        return "exit"   # This maps to END

print("✅ All node functions defined: greeting_node → random_number_node ↺ (should_continue_node)")

In [ ]:
# 💡 The execution flow we're building:
# greeting → random_number → random_number → ... (repeat until counter >= 5) → END
# This is a SELF-LOOP pattern — the random_number node loops back to itself!

In [ ]:
# 🏗️ Build the Looping Graph

graph = StateGraph(AgentState)

# Register nodes
graph.add_node("greeting", greeting_node)          # Setup node — runs once
graph.add_node("random_number", random_number_node)  # Loop body — runs multiple times

# Wire the sequential part: greeting → random_number
graph.add_edge("greeting", "random_number")

# 🔄 THE KEY PART: Add conditional edges that create the loop!
# After random_number runs, call should_continue_node to decide what happens next
graph.add_conditional_edges(
    source="random_number",        # After this node finishes...
    path=should_continue_node,     # ...call this function to decide next step
    path_map={
        # "loop" → go back to random_number (SELF-LOOP — this creates the cycle!)
        "loop": "random_number",
        # "exit" → go to END (breaks out of the loop)
        "exit": END
    }
)

# Set the entry point — greeting always runs first
graph.set_entry_point("greeting")

# Compile the graph
app = graph.compile()

print("✅ Looping graph compiled! Flow: START → greeting → random_number ↺ (loop until counter >= 5) → END")

In [ ]:
# 🎨 Visualize the looping graph — look for the self-loop arrow!
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))
# 👆 You should see: __start__ → greeting → random_number → (loops back to itself OR → __end__)

### 🚀 Running the Looping Graph

Let's run the loop! The graph will:
1. Greet the user (once)
2. Generate a random number and increment the counter (loop body)
3. Check if counter < 5 — if yes, loop back to step 2; if no, exit

Watch the counter increment with each iteration until it reaches 5.

In [ ]:
# 📥 Create the initial state — empty number list, counter starts at 0
initial_state = AgentState(name="Mahendra", number=[], counter=0)
print(f"📥 Initial state: {initial_state}")

In [ ]:
# 🚀 Run the graph — watch the loop execute 5 times!
# ⚠️ Note: We print initial_state AFTER invoke — but app.invoke() returns a NEW dict,
#    so initial_state itself may or may not be modified (depends on implementation)
app.invoke(initial_state)
print(f"\n📤 initial_state after invoke: {initial_state}")
# 💡 The returned result (not captured here) is the authoritative final state

In [ ]:
# ✅ Best practice: Always capture the return value of app.invoke()
# The returned dict is the authoritative final state after all nodes have executed
result = app.invoke(initial_state)
print(f"\n📤 Final result: {result}")
print(f"🎯 Generated {len(result['number'])} random numbers: {result['number']}")
print(f"🔄 Total loop iterations: {result['counter']}")

### ✅ Looping Graph — Section Summary

You just built a graph with **iterative execution**! The self-loop pattern:

1. **Setup node** — runs once to initialize state (greeting + counter reset)
2. **Loop body node** — runs on each iteration (generates random number, increments counter)
3. **Termination check** — a path function that returns `"loop"` to continue or `"exit"` to stop
4. **Self-loop edge** — in the path_map, one path points back to the same node: `"loop": "random_number"`

> 🧠 **Critical safety rule**: Always include a termination condition! Without the counter check, the loop would run forever. Common termination strategies: counter limits, convergence checks, or external signals.

---

## 🧪 Exercise: Looping Graph

### Your Task — Automatic Higher/Lower Guessing Game

Build a looping graph that plays the **Higher or Lower** game automatically!

**Rules:**
- Set the bounds to between **1 to 20** — the graph picks a secret number in this range
- The graph keeps guessing (max **7 guesses**) — if the guess is correct, it stops; if not, it keeps looping
- Each time a number is guessed, the **hint node** should say "higher" or "lower"
- The graph should **adjust its bounds** after every guess based on the hint

**Input**: 
```python
{"player_name": "Student", "guesses": [], "attempts": 0, "lower_bound": 1, "upper_bound": 20}
```

### 💡 Hints

- You'll need nodes for: `setup`, `guess`, `hint`, and a conditional edge for `should_continue`
- The `guess` node should pick a number between `lower_bound` and `upper_bound`
- The `hint` node should compare the guess to the secret number and update `lower_bound` or `upper_bound`
- The conditional edge checks: correct guess → `"exit"`, wrong guess AND attempts < 7 → `"loop"`, max attempts reached → `"exit"`

![Exercise_Looping_graph](Exercise_Looping_graph.png)

# 🧠 Adding Memory to Agents

### 📚 What You'll Learn

- How to implement **short-term memory** (conversation history within a session)
- How to implement **long-term memory** (user preferences persisted to disk)
- How to build a **multi-node pipeline** that processes, learns, and responds
- How memory enables **context-aware responses** that improve over time

### 🧠 Key Concepts

- **Short-Term Memory**: Conversation history stored in the state's `conversation_history` list — lives only during the session
- **Long-Term Memory**: User preferences persisted to a JSON file on disk — survives across sessions
- **Memory Processor Node**: Stores each user message in conversation history with timestamps
- **Preference Learner Node**: Detects preference keywords ("favorite", "love", "like") and saves them
- **Response Generator Node**: Uses conversation history to generate context-aware responses

### 🎯 Why This Matters

Without memory, every agent interaction starts from scratch. Memory transforms a stateless tool into a **personalized assistant** that:
- Remembers what you discussed earlier in the conversation
- Learns your preferences over time
- Provides increasingly relevant and contextual responses

### 🔄 Pipeline: Memory → Learn → Respond

```
START → memory_processor → preference_learner → response_generator → END
         (store msg)        (detect prefs)       (use history)
```

---

In [ ]:
# 📦 Import everything we need for the memory-enabled agent
from langgraph.graph import StateGraph, START, END  # Graph building blocks
from typing import TypedDict, List, Dict             # Type annotations for complex state
from pathlib import Path                              # File system paths (cross-platform)
import json                                           # For persisting preferences to disk
import datetime                                       # For timestamping conversation entries

print("✅ Imports loaded for memory-enabled agent")

In [ ]:
# 📋 Define the ConversationState — our most complex state schema yet!
# This state has BOTH short-term memory (conversation_history) and long-term memory (user_preferences)

class ConversationState(TypedDict):
    user_name: str                              # The user's name
    conversation_history: List[Dict[str, str]]  # 🧠 SHORT-TERM MEMORY: list of timestamped messages
    current_message: str                        # The user's current input message
    response: str                               # The agent's generated response
    user_preferences: List[str]                 # 💾 LONG-TERM MEMORY: learned preferences (persisted to disk)
    session_count: int                          # Tracks total number of interactions in this session

# 💡 conversation_history is append-only within a session — it grows with each interaction
# 💡 user_preferences is persisted to a JSON file — it survives across notebook restarts!
print("✅ ConversationState defined with short-term and long-term memory fields")

In [ ]:
# 🔧 Define helper functions and all three node functions for the memory pipeline

# --- Long-Term Memory Persistence Helpers ---
# These functions read/write user preferences to a JSON file on disk
PREFERENCES_FILE = Path("user_preferences.json")  # File path for persisted preferences

# Keywords that indicate a user is expressing a preference
PREFERENCE_KEYWORDS = ["favorite", "favourite", "prefer", "preference", "love", "like"]


def load_user_preferences() -> List[str]:
    """Load previously saved user preferences from disk (long-term memory)."""
    # If the file doesn't exist yet, return an empty list
    if not PREFERENCES_FILE.exists():
        return []

    try:
        # Read the JSON file
        with PREFERENCES_FILE.open("r", encoding="utf-8") as file:
            data = json.load(file)

        # Handle the normal case: data is a list of preference strings
        if isinstance(data, list):
            return [str(item) for item in data]

        # Backward compatibility: handle older dict format gracefully
        if isinstance(data, dict):
            return [f"{key}: {value}" for key, value in data.items()]
    except (json.JSONDecodeError, OSError):
        pass  # If file is corrupted, start fresh

    return []


def save_user_preferences(preferences: List[str]) -> None:
    """Persist user preferences to disk (long-term memory write)."""
    try:
        with PREFERENCES_FILE.open("w", encoding="utf-8") as file:
            json.dump(preferences, file, indent=2)  # Pretty-print for readability
    except OSError:
        print("⚠️ Warning: Unable to persist user preferences.")


# --- Node 1: Memory Processor (Short-Term Memory) ---
def memory_processor_node(state: ConversationState) -> ConversationState:
    """
    SHORT-TERM MEMORY NODE: Stores the current message in conversation history.
    Each entry includes a timestamp, the user's message, and a type tag.
    """
    # Append current message to conversation history with metadata
    state["conversation_history"].append({
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),  # When the message was received
        "user": state["current_message"],  # The actual message content
        "type": "user_input"               # Tag to distinguish user messages from agent responses
    })

    # Increment session counter
    state["session_count"] += 1

    print(f"  🧠 Memory updated. Total interactions: {len(state['conversation_history'])}")
    return state


# --- Node 2: Response Generator (Uses Short-Term Memory) ---
def response_generator_node(state: ConversationState) -> ConversationState:
    """
    RESPONSE NODE: Generates context-aware responses using conversation history.
    The response changes based on how many interactions have occurred (memory depth).
    """
    # Check how many messages are in our conversation history
    history_count = len(state["conversation_history"])

    # Generate different responses based on conversation depth
    if history_count == 1:
        # First interaction — introduce yourself
        state["response"] = f"Hello {state['user_name']}! Nice to meet you. How can I help you today?"
    elif history_count <= 3:
        # Early conversation — acknowledge we have some history
        state["response"] = f"I remember we've talked {history_count} times. What would you like to discuss now?"
    else:
        # Deeper conversation — reference specific previous topics from memory!
        # Pull the last few user messages from conversation history
        prev_topics = [msg["user"] for msg in state["conversation_history"][-3:] if msg["type"] == "user_input"]
        state["response"] = f"We've had {history_count} interactions. Recently you mentioned: {', '.join(prev_topics[-2:])}. What's next?"

    # Store the agent's response in conversation history too (for complete records)
    state["conversation_history"].append({
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "agent": state["response"],
        "type": "agent_response"
    })

    return state


# --- Node 3: Preference Learner (Long-Term Memory) ---
def preference_learner_node(state: ConversationState) -> ConversationState:
    """
    LONG-TERM MEMORY NODE: Detects preference keywords in user messages and saves them.
    Preferences are persisted to disk so they survive across sessions.
    """
    # First, load any previously saved preferences from disk
    loaded_preferences = load_user_preferences()

    # Merge loaded preferences with in-memory preferences (avoid duplicates)
    merged_preferences = list(loaded_preferences)
    for pref in state["user_preferences"]:
        if pref not in merged_preferences:
            merged_preferences.append(pref)
    state["user_preferences"] = merged_preferences

    # Check if the current message contains any preference keywords
    message = state["current_message"].strip()
    normalized_message = message.lower()  # Case-insensitive matching

    if any(keyword in normalized_message for keyword in PREFERENCE_KEYWORDS):
        # Found a preference! Store the original (non-lowered) message
        user_preference = message
        if user_preference not in state["user_preferences"]:
            state["user_preferences"].append(user_preference)
            save_user_preferences(state["user_preferences"])  # 💾 Persist to disk!
            print(f"  💾 New preference saved: {user_preference}")
        else:
            print(f"  ℹ️ Preference already stored: {user_preference}")
    elif len(state["user_preferences"]) > 0:
        print(f"  📋 Current preferences: {state['user_preferences']}")

    return state

print("✅ All memory node functions defined: memory_processor → preference_learner → response_generator")

### 🏗️ Building and Compiling the Memory Graph

Now we wire the three nodes together in a sequential pipeline. The order matters:
1. **memory_processor** runs first — stores the incoming message
2. **preference_learner** runs second — checks for preference keywords
3. **response_generator** runs last — uses the full history to craft a response

In [ ]:
# 🏗️ Build the Memory-Enabled Conversation Graph
# Pipeline: memory_processor → preference_learner → response_generator

memory_graph = StateGraph(ConversationState)

# Register all three nodes
memory_graph.add_node("memory_processor", memory_processor_node)     # Step 1: Store message in history
memory_graph.add_node("preference_learner", preference_learner_node) # Step 2: Detect and save preferences
memory_graph.add_node("response_generator", response_generator_node) # Step 3: Generate context-aware response

# Wire the sequential pipeline: START → memory → preferences → response → END
memory_graph.add_edge(START, "memory_processor")
memory_graph.add_edge("memory_processor", "preference_learner")
memory_graph.add_edge("preference_learner", "response_generator")
memory_graph.add_edge("response_generator", END)

# Compile the graph
memory_app = memory_graph.compile()

print("✅ Memory graph compiled! Flow: START → memory_processor → preference_learner → response_generator → END")

In [ ]:
# 🎨 Visualize the memory pipeline
from IPython.display import Image, display
display(Image(memory_app.get_graph().draw_mermaid_png()))
# 👆 You should see: __start__ → memory_processor → preference_learner → response_generator → __end__

### 🧪 Testing the Memory Agent — Multi-Turn Conversation

Now let's test our memory agent with **four consecutive interactions** to see how memory accumulates:

1. **First interaction** — A simple greeting (no preferences)
2. **Second interaction** — Contains "favorite" keyword → preference learner should detect it!
3. **Third interaction** — Contains "love" keyword → another preference saved!
4. **Fourth interaction** — Agent should reference previous conversation topics from memory

> 💡 **Key insight**: We pass the result of each `invoke()` back as input for the next one. This is how **short-term memory persists** — the `conversation_history` list grows with each interaction.

In [ ]:
# 🚀 Test the memory-enabled agent with multiple interactions
# First, set up the initial conversation state

conversation_state = ConversationState(
    user_name="Alice",                            # The user we're chatting with
    conversation_history=[],                       # Empty history — brand new conversation
    current_message="Hello, I'm new here",        # First message from the user
    response="",                                   # Will be filled by response_generator
    user_preferences=load_user_preferences(),      # Load any previously saved preferences from disk
    session_count=0                                # No interactions yet
)

print(f"📂 Loaded saved preferences: {conversation_state['user_preferences']}")
print()

# === First Interaction ===
# This is the first message — the agent should introduce itself
print("=" * 50)
print("=== 💬 First Interaction ===")
print("=" * 50)
result1 = memory_app.invoke(conversation_state)
save_user_preferences(result1["user_preferences"])  # Persist any new preferences
print(f"🤖 Agent: {result1['response']}")
print(f"📊 Memory size: {len(result1['conversation_history'])} entries")
print()

In [ ]:
# 🔄 Continue the conversation — pass the SAME state to maintain memory continuity!
# Each subsequent invoke carries forward the conversation_history from the previous result

# === Second Interaction ===
# This message contains a preference keyword ("favorite") — the preference learner should detect it!
result1["current_message"] = "My favorite color is blue"
print("=" * 50)
print("=== 💬 Second Interaction ===")
print("=" * 50)
result2 = memory_app.invoke(result1)
save_user_preferences(result2["user_preferences"])
print(f"🤖 Agent: {result2['response']}")
print(f"📊 Memory size: {len(result2['conversation_history'])} entries")
print()

# === Third Interaction ===
# Another preference keyword ("love") — this should also be saved!
result2["current_message"] = "I also love pizza"
print("=" * 50)
print("=== 💬 Third Interaction ===")
print("=" * 50)
result3 = memory_app.invoke(result2)
save_user_preferences(result3["user_preferences"])
print(f"🤖 Agent: {result3['response']}")
print(f"📊 Memory size: {len(result3['conversation_history'])} entries")
print()

# === Fourth Interaction ===
# No preference keyword here — but the agent should reference previous topics from memory!
result3["current_message"] = "What did we discuss before?"
print("=" * 50)
print("=== 💬 Fourth Interaction ===")
print("=" * 50)
result4 = memory_app.invoke(result3)
save_user_preferences(result4["user_preferences"])
print(f"🤖 Agent: {result4['response']}")
print(f"📊 Final memory size: {len(result4['conversation_history'])} entries")
print(f"💾 User preferences learned: {result4['user_preferences']}")
print()
print("🎯 Notice how the agent's responses became more contextual as memory grew!")

---

## 🎓 Notebook Summary — What You've Accomplished

Congratulations! You've built **four increasingly complex LangGraph patterns**:

| Pattern | What You Learned | Key LangGraph Feature |
|---|---|---|
| **TypedDict** | Type-safe state definitions | `TypedDict` as the state schema |
| **Sequential Graph** | Linear multi-step pipelines | `add_edge()` for sequential flow |
| **Conditional Graph** | Runtime branching based on state | `add_conditional_edges()` with path functions |
| **Looping Graph** | Iterative execution with termination | Self-loop edges in `path_map` |
| **Memory Agent** | Short-term and long-term memory | State accumulation + file persistence |

### 🔑 Key Takeaways

1. **Every LangGraph app starts with a TypedDict** — define your state schema first
2. **Nodes are just functions** — they take state in and return state out
3. **Edges define flow** — sequential (`add_edge`), conditional (`add_conditional_edges`), or looping (self-referencing path_map)
4. **Always compile before running** — `graph.compile()` validates your graph structure
5. **Memory is just state management** — short-term lives in the state, long-term gets persisted externally

### ⏭️ Next Steps

- Complete the exercises above to practice each pattern
- Try combining patterns (e.g., a conditional graph with memory, or a loop with conditional exits)
- Explore LangGraph's built-in checkpointing for more robust memory management